# Common functions
Reusable helpers shared by all pipeline notebooks: column cleanup, value parsing, ingestion metadata, deduplication, Delta table writes and a data quality framework. Load with `%run` after `config/pipeline_config`.

In [0]:
import re
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F


def sanitize_column_names(df: DataFrame) -> DataFrame:
    """Remove spaces/special characters so names are valid Delta columns ('Unit Price' -> 'UnitPrice')."""
    return df.toDF(*[re.sub(r"[^0-9A-Za-z_]", "", c) for c in df.columns])


def add_ingestion_metadata(df: DataFrame) -> DataFrame:
    """Add load timestamp and source file path. Call directly on a file-based DataFrame."""
    return (df
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file_name", F.col("_metadata.file_path")))


def to_int(c: str):
    return F.expr(f"try_cast(trim(`{c}`) AS INT)")


def clean_string(c: str):
    """Trim and turn empty strings into NULL."""
    return F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))


def parse_money(c: str):
    """'$2,024.99' -> 2024.99 (NULL when not parseable)."""
    return F.expr(f"try_cast(regexp_replace(`{c}`, '[$,]', '') AS DOUBLE)")


def parse_long_date(c: str):
    """'Friday, August 25, 2017' -> 2017-08-25 (NULL when not parseable)."""
    return F.expr(f"try_to_date(regexp_replace(trim(`{c}`), '^[A-Za-z]+, ', ''), 'MMMM d, yyyy')")


def deduplicate(df: DataFrame, keys: list, order_col: str = "ingestion_timestamp") -> DataFrame:
    """Keep the latest row per primary key."""
    w = Window.partitionBy(*keys).orderBy(F.col(order_col).desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

In [0]:
def write_delta_table(df: DataFrame, table_name: str, path: str, mode: str = "overwrite") -> int:
    """Write df as Delta to an ADLS path and register it as an external Unity Catalog table.

    Fails if the table already exists at a different location, so data never lands where the table cannot see it.
    Returns the row count of the registered table.
    """
    writer = df.write.format("delta").mode(mode)
    if mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    writer.save(path)

    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{path}'")

    location = spark.sql(f"DESCRIBE DETAIL {table_name}").first()["location"]
    if location.rstrip("/") != path.rstrip("/"):
        raise ValueError(f"{table_name} points to {location}, expected {path}")

    rows = spark.table(table_name).count()
    print(f"✓ {table_name}: {rows} rows -> {path}")
    return rows

In [0]:
class DataQualityChecker:
    """Collects check results. Critical failures stop the pipeline; non-critical ones are reported as WARN."""

    def __init__(self, layer: str):
        self.layer = layer
        self.results = []

    def check(self, table: str, name: str, failed_rows: int, critical: bool = True):
        status = "PASS" if failed_rows == 0 else ("FAIL" if critical else "WARN")
        self.results.append((self.layer, table, name, int(failed_rows), status))

    def not_empty(self, table: str, df: DataFrame):
        self.check(table, "table is not empty", 0 if df.limit(1).count() else 1)

    def not_null(self, table: str, df: DataFrame, columns: list, critical: bool = True):
        for c in columns:
            self.check(table, f"{c} is not null", df.filter(F.col(c).isNull()).count(), critical)

    def unique(self, table: str, df: DataFrame, keys: list):
        self.check(table, f"{'+'.join(keys)} is unique", df.count() - df.select(*keys).distinct().count())

    def rule(self, table: str, name: str, df: DataFrame, failing_condition, critical: bool = True):
        """failing_condition: Column expression that is TRUE for bad rows."""
        self.check(table, name, df.filter(failing_condition).count(), critical)

    def referential(self, table: str, child: DataFrame, parent_name: str, parent: DataFrame, key, critical: bool = False):
        keys = key if isinstance(key, list) else [key]
        orphans = child.dropna(subset=keys).join(parent.select(*keys).distinct(), keys, "left_anti").count()
        self.check(table, f"{'+'.join(keys)} exists in {parent_name}", orphans, critical)

    def results_df(self) -> DataFrame:
        return (spark.createDataFrame(self.results, "layer STRING, table_name STRING, check_name STRING, failed_rows LONG, status STRING")
                .withColumn("checked_at", F.current_timestamp()))

    def assert_no_critical_failures(self):
        failed = [f"{r[1]}: {r[2]} ({r[3]} rows)" for r in self.results if r[4] == "FAIL"]
        passed = sum(r[4] == "PASS" for r in self.results)
        warned = sum(r[4] == "WARN" for r in self.results)
        print(f"{self.layer} DQ: {passed} passed, {warned} warnings, {len(failed)} critical failures")
        if failed:
            raise ValueError("Critical data quality checks failed:\n" + "\n".join(failed))